## 1 - Dataset Acquisition and Initial Setup

In this first step, we are downloading the CelebDF dataset, the **3rd** version.

In [16]:
%pip install "opencv-python<5" pandas numpy tqdm pillow matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import cv2
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [4]:
# Raw Celeb-DF-v3 videos live in the shared Utility/ folder (same place as
# Utility/FaceDetection/, used in the next notebook of this pipeline), not on an
# external drive -- copy/extract the dataset there before running this notebook.
UTILITY_DIR = Path.cwd().parent.parent / "Utility"  # Datasets/ -> 02_Extended_Framework/ -> deepfake-forensics-pipeline/Utility
CELEB_HD_PATH = str(UTILITY_DIR / "Celeb-DF-v3") + "/"

base_path = CELEB_HD_PATH
folders = os.listdir(base_path)
print("Contents inside", base_path, ": ")
print('\n'.join([f"- {f}" for f in folders]))

Contents inside /home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/Utility/Celeb-DF-v3/ : 
- Celeb-synthesis
- YouTube-real
- List_of_testing_videos.txt
- Celeb-real
- .DS_Store


In [5]:
for folder in folders:
    if folder.startswith('._'):
        continue
        
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        try:
            files = [f for f in os.listdir(folder_path) if not f.startswith('._')]
            if files:
                print(f"Directory '{folder}': {len(files)} files (e.g., {files[0]})")
            else:
                print(f"Directory '{folder}': Empty")
        except Exception as e:
            print(f"Error accessing {folder}: {e}")
    else:
        print(f"File found: {folder}")

Directory 'Celeb-synthesis': 4 files (e.g., FaceReenact)
Directory 'YouTube-real': 300 files (e.g., 00276.mp4)
File found: List_of_testing_videos.txt
Directory 'Celeb-real': 590 files (e.g., id36_0008.mp4)
File found: .DS_Store


In [6]:
synthesis_path = os.path.join(base_path, 'Celeb-synthesis')
all_fake_videos = []

print("Looking for fake videos...")

for root, dirs, files in os.walk(synthesis_path):
    vids = [os.path.join(root, f) for f in files 
            if f.endswith(('.mp4', '.avi')) and not f.startswith('._')]
    all_fake_videos.extend(vids)

print(f"\n--- FAKE VIDEO ---")
print(f"Total: {len(all_fake_videos)}")

if all_fake_videos:
    print(f"Example path: {all_fake_videos[0]}")
    counts = {}
    for path in all_fake_videos:
        parts = path.split(os.sep)
        idx = parts.index('Celeb-synthesis')
        category = parts[idx + 1]
        counts[category] = counts.get(category, 0) + 1
    
    for cat, count in counts.items():
        print(f"- {cat}: {count} video")

Looking for fake videos...

--- FAKE VIDEO ---
Total: 53196
Example path: /home/lucia_pola/internship-deepfake-forensic/deepfake-forensics-pipeline/Utility/Celeb-DF-v3/Celeb-synthesis/FaceReenact/LivePortrait/id31_id35_0002.mp4
- FaceReenact: 13632 video
- TalkingFace: 20279 video
- FaceSwap: 19285 video


In [7]:
real_folders = ['Celeb-real', 'YouTube-real']

print(f"--- REAL VIDEO ---")

for folder in real_folders:
    folder_path = os.path.join(base_path, folder)
    print(f"\nDirectory: {folder}")
    
    if os.path.exists(folder_path):
        all_items = [f for f in os.listdir(folder_path) if not f.startswith('._')]
        
        vids = [f for f in all_items if f.endswith(('.mp4', '.avi'))]
        subdirs = [f for f in all_items if os.path.isdir(os.path.join(folder_path, f))]
        
        print(f"  - Total elements: {len(all_items)}")
        print(f"  - Total video: {len(vids)}")
        print(f"  - Subdirectory: {len(subdirs)}")
        
        if vids:
            print(f"  - Example video: {vids[:3]}")
        if subdirs:
            print(f"  - Example subdirectory: {subdirs[:3]}")
            first_sub = os.path.join(folder_path, subdirs[0])
            inner_vids = [f for f in os.listdir(first_sub) if f.endswith(('.mp4', '.avi'))]
            print(f"  - Video inside the first subdirectory ({subdirs[0]}): {len(inner_vids)}")
    else:
        print(f"  Direcotry not founded: {folder_path}")

--- REAL VIDEO ---

Directory: Celeb-real
  - Total elements: 590
  - Total video: 590
  - Subdirectory: 0
  - Example video: ['id36_0008.mp4', 'id10_0008.mp4', 'id19_0006.mp4']

Directory: YouTube-real
  - Total elements: 300
  - Total video: 300
  - Subdirectory: 0
  - Example video: ['00276.mp4', '00147.mp4', '00069.mp4']


## 2 - Pandas DataFrame

This section creates a single pandas DataFrame containing all videos from the Celeb-DF-v3 dataset.

**Columns:**
- `video`: the video filename (e.g., with `.mp4` extension).
- `full_path`: the complete file path to the video on the disk.
- `label`: 0 = real/original, 1 = fake/synthesis.
- `dataset`: the name of the dataset (e.g., "Celeb-DF-v3").
- `category`: the specific subfolder or subset category (e.g., 'Celeb-real', 'YouTube-real', or synthesis sub-categories).
- `method`: the specific deepfake generation method used, or "original" for real videos.
- `target`: ID of the target subject (the person whose face is being replaced or modified).
- `source`: ID of the source subject (the person providing the new face). For real videos, this is identical to the target.
- `sequence`: the video sequence identifier extracted from the filename, or "original" for real videos.

In [8]:
data = []

real_folders = ['Celeb-real', 'YouTube-real']
for folder in real_folders:
    folder_path = os.path.join(base_path, folder)
    if not os.path.exists(folder_path): continue
    
    for video in os.listdir(folder_path):
        if video.endswith('.mp4') and not video.startswith('._'):
            name = video.replace('.mp4', '')
            target = name.split('_')[0] if 'id' in name else name
            
            data.append({
                "video": video,
                "label": 0,
                "dataset": "Celeb-DF-v3",
                "category": folder,
                "method": "original",
                "target": target,
                "source": target, 
                "sequence": "original",
                "full_path": os.path.join(folder_path, video)
            })

synthesis_path = os.path.join(base_path, 'Celeb-synthesis')
if os.path.exists(synthesis_path):
    for root, dirs, files in os.walk(synthesis_path):
        for video in files:
            if video.endswith('.mp4') and not video.startswith('._'):
                name = video.replace('.mp4', '')
                parts = name.split('_')
   
                path_parts = root.split(os.sep)
                idx = path_parts.index('Celeb-synthesis')
                category = path_parts[idx + 1] if len(path_parts) > idx + 1 else "Unknown"
                method = path_parts[idx + 2] if len(path_parts) > idx + 2 else "Unknown"

                source = parts[0] if len(parts) > 0 else "Unknown"
                sequence = parts[1] if len(parts) > 1 else "Unknown"

                target = "Unknown"
                for p in parts:
                    if p.startswith('id') and p != source:
                        target = p
                        break
                
                if target == "Unknown":
                    target = source

                data.append({
                    "video": video,
                    "label": 1,
                    "dataset": "Celeb-DF-v3",
                    "category": category,
                    "method": method,
                    "target": target,
                    "source": source,
                    "sequence": sequence,
                    "full_path": os.path.join(root, video)
                })

df_celeb = pd.DataFrame(data)

print("--- CELEB-DF-V3 DATAFRAME ---")
display(df_celeb.sample(15))

--- CELEB-DF-V3 DATAFRAME ---


,video,label,dataset,category,method,target,source,sequence,full_path
30458,id20_0009_test_id08701_A2nebxHmWOU.mp4,1,Celeb-DF-v3,TalkingFace,IP_LAP,id08701,id20,0009,/home/lucia_pola/internship-deepfake-forensic/...
19320,id17_0004_test_id05124_akdL5HB5LAA.mp4,1,Celeb-DF-v3,TalkingFace,Real3DPortrait,id05124,id17,0004,/home/lucia_pola/internship-deepfake-forensic/...
8862,id12_id7_0001.mp4,1,Celeb-DF-v3,FaceReenact,MCNET,id7,id12,id7,/home/lucia_pola/internship-deepfake-forensic/...
50101,id19_id20_0007.mp4,1,Celeb-DF-v3,FaceSwap,InSwapper,id20,id19,id20,/home/lucia_pola/internship-deepfake-forensic/...
5341,id3_id6_0005.mp4,1,Celeb-DF-v3,FaceReenact,DaGAN,id6,id3,id6,/home/lucia_pola/internship-deepfake-forensic/...
35342,id30_id29_0008.mp4,1,Celeb-DF-v3,FaceSwap,MobileFaceSwap,id29,id30,id29,/home/lucia_pola/internship-deepfake-forensic/...
37198,id22_id21_0002.mp4,1,Celeb-DF-v3,FaceSwap,UniFace,id21,id22,id21,/home/lucia_pola/internship-deepfake-forensic/...
49299,id16_id28_0009.mp4,1,Celeb-DF-v3,FaceSwap,InSwapper,id28,id16,id28,/home/lucia_pola/internship-deepfake-forensic/...
40021,id60_id59_0000.mp4,1,Celeb-DF-v3,FaceSwap,GHOST,id59,id60,id59,/home/lucia_pola/internship-deepfake-forensic/...
47820,id38_id21_0004.mp4,1,Celeb-DF-v3,FaceSwap,HifiFace,id21,id38,id21,/home/lucia_pola/internship-deepfake-forensic/...


### 2.1 - Sanity Check

Quick checks to ensure the dataset is loaded correctly and to understand its composition:

- **Total number of videos:** Overall count of files found on the drive.
- **Label distribution:** Balance between real (0) and fake (1) videos.
- **Method distribution:** The top AI manipulation methods used to generate the fakes.
- **Category distribution:** Breakdown by subset (e.g., Celeb-real, YouTube-real, Celeb-synthesis).
- **Identity analysis:** Counts of unique target and source IDs, including the overlap of identities present in both real and fake sets.

In [9]:
print(f"--- CELEB-DF-V3 AUDIT ---")
print(f"Total videos found on HDD: {len(df_celeb)}")

print("\nLabel Distribution:")
print(df_celeb["label"].value_counts().rename({0: '0 (Real)', 1: '1 (Fake)'}))

print("\nAI Method Distribution (Top 10):")
print(df_celeb["method"].value_counts().head(10))

print("\nCategory Distribution:")
print(df_celeb["category"].value_counts())

print("\nIdentity Analysis (Target-based):")
unique_targets = df_celeb['target'].nunique()
print(f"Total unique Target identities: {unique_targets}")

real_targets = set(df_celeb[df_celeb['label'] == 0]['target'])
fake_targets = set(df_celeb[df_celeb['label'] == 1]['target'])
overlap = real_targets.intersection(fake_targets)

print(f"Targets present in both Real and Fake: {len(overlap)}")
print(f"Targets only in Real: {len(real_targets - fake_targets)}")
print(f"Targets only in Fake: {len(fake_targets - real_targets)}")

if df_celeb['label'].sum() > 0:
    unique_sources = df_celeb[df_celeb['label'] == 1]['source'].nunique()
    print(f"Unique Sources used for fakes: {unique_sources}")

--- CELEB-DF-V3 AUDIT ---
Total videos found on HDD: 54086

Label Distribution:
label
1 (Fake)    53196
0 (Real)      890
Name: count, dtype: int64

AI Method Distribution (Top 10):
method
Celeb-DF-v2       5639
SadTalker         2950
FLOAT             2950
AniTalker         2950
IP_LAP            2935
EchoMimic         2885
EDTalk            2865
Real3DPortrait    2744
MobileFaceSwap    1953
SimSwap           1953
Name: count, dtype: int64

Category Distribution:
category
TalkingFace     20279
FaceSwap        19285
FaceReenact     13632
Celeb-real        590
YouTube-real      300
Name: count, dtype: int64

Identity Analysis (Target-based):
Total unique Target identities: 476
Targets present in both Real and Fake: 57
Targets only in Real: 302
Targets only in Fake: 117
Unique Sources used for fakes: 59


### 2.2  - Distribution of Fake Videos per Target

Analyzes how many fake videos exist per target identity to identify if some identities dominate the fake samples

In [10]:
fake_df = df_celeb[df_celeb["label"] == 1]
per_target = fake_df.groupby("target").size()
print("\n--- FAKE PER TARGET STATISTICS ---")
print(per_target.describe())


--- FAKE PER TARGET STATISTICS ---
count     174.000000
mean      305.724138
std       286.455746
min         7.000000
25%       123.750000
50%       218.000000
75%       370.500000
max      1318.000000
dtype: float64


### 2.3 - Distribution Of Methods per Target (Check Variability):

Creates a pivot table showing how many videos of each manipulation method exist per target to verify that each identity has a representative set of manipulation methods and to detect identities with too few or missing manipulation types

In [11]:
pivot_celeb = pd.pivot_table(
    df_celeb,
    index="target",
    columns="method",
    values="video",
    aggfunc="count",
    fill_value=0
)

print("\n--- TOP 20 TARGETS BY METHOD COVERAGE ---")
if 'original' in pivot_celeb.columns:
    display(pivot_celeb.sort_values(by='original', ascending=False).head(20))
else:
    display(pivot_celeb.sample(20))


--- TOP 20 TARGETS BY METHOD COVERAGE ---


method,AniTalker,BlendFace,Celeb-DF-v2,DaGAN,EDTalk,EchoMimic,FLOAT,FSRT,GHOST,HifiFace,...,LIA,LivePortrait,MCNET,MobileFaceSwap,Real3DPortrait,SadTalker,SimSwap,TPSMM,UniFace,original
target,,,,,,,,,,,,,,,,,,,,,
id13,0,6,16,6,0,0,0,6,6,6,...,6,6,6,6,0,0,6,6,6,16
id16,0,48,145,48,0,0,0,48,48,48,...,48,48,48,48,0,0,48,48,48,14
id11,0,6,16,5,0,0,0,5,6,6,...,6,6,5,6,0,0,6,6,6,11
id25,0,17,71,17,0,0,0,17,17,17,...,17,17,17,17,0,0,17,17,17,11
id1,0,55,158,55,0,0,0,55,55,55,...,55,55,55,55,0,0,55,55,55,10
id28,0,78,226,78,0,0,0,78,78,78,...,78,78,78,78,0,0,78,78,78,10
id27,0,25,64,20,0,0,0,20,25,25,...,20,20,20,25,0,0,25,20,25,10
id26,0,78,217,78,0,0,0,78,78,78,...,78,78,78,78,0,0,78,78,78,10
id0,0,43,125,43,0,0,0,43,43,43,...,43,43,43,43,0,0,43,43,43,10


### 2.4 - Check Label Balance

Checks the ratio of real to fake videos across the entire dataset to understand dataset imbalance

In [12]:
print("Normalize label ditribution:")
print(df_celeb["label"].value_counts(normalize=True))

Normalize label ditribution:
label
1    0.983545
0    0.016455
Name: proportion, dtype: float64


## 3 - Metadata Extraction (OpenCV)

In this section, we use **OpenCV (`cv2`)** to scan through the entire Celeb-DF-v3 dataset and extract key metadata:
- `total_frames`
- `fps`
- `duration_sec`
- `resolution`

In [13]:
tqdm.pandas(desc="Extracting Celeb-DF-v3 metadata")

def get_video_metadata(video_path):
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return pd.Series([None, None, None, None])
    
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        duration = total_frames / fps if fps > 0 else 0
        
        cap.release()
        return pd.Series([total_frames, fps, duration, f"{width}x{height}"])
    
    except Exception as e:
        return pd.Series([None, None, None, None])

print(f"Scanning {len(df_celeb)} videos. This will take a few minutes...")

df_celeb[['total_frames', 'fps', 'duration_sec', 'resolution']] = \
    df_celeb['full_path'].progress_apply(get_video_metadata)

print("\nMetadata extraction complete!")

display(df_celeb[['video', 'total_frames', 'duration_sec', 'resolution', 'method']].sample(20))

Scanning 54086 videos. This will take a few minutes...


Extracting Celeb-DF-v3 metadata: 100%|██| 54086/54086 [03:51<00:00, 234.07it/s]


Metadata extraction complete!


,video,total_frames,duration_sec,resolution,method
22895,id30_0007_test_id05176_YbxM5PV2s1U.mp4,124,4.960000,256x256,EDTalk
6522,id1_id16_0000.mp4,258,8.600000,256x256,DaGAN
22192,id61_0006_test_id01541_2P7hzPq5iDw.mp4,166,6.640000,256x256,EDTalk
27815,id7_0004_test_id06811_Z8pDqRv3I8c.mp4,132,5.500000,512x512,EchoMimic
45892,id56_id55_0007.mp4,235,8.103444,856x478,Celeb-DF-v2
39542,id28_id34_0001.mp4,333,11.100000,846x474,GHOST
51611,id61_id5_0000.mp4,420,14.000000,856x478,SimSwap
6015,id38_id23_0009.mp4,315,10.500000,256x256,DaGAN
50322,id47_id48_0001.mp4,316,10.533333,830x470,SimSwap
9344,id1_id4_0007.mp4,445,14.833333,256x256,MCNET


In [14]:
print("\n--- VIDEO DURATION STATISTICS (in seconds) ---")
print(df_celeb['duration_sec'].describe())

print("\n--- VIDEO FRAMES STATISTICS ---")
print(df_celeb['total_frames'].describe())

display(df_celeb.sample(20))


--- VIDEO DURATION STATISTICS (in seconds) ---
count    54086.000000
mean        10.308581
std          4.042332
min          0.033333
25%          6.720000
50%         10.400000
75%         13.000000
max         36.950000
Name: duration_sec, dtype: float64

--- VIDEO FRAMES STATISTICS ---
count    54086.000000
mean       287.973727
std        122.839820
min          1.000000
25%        170.000000
50%        306.000000
75%        365.000000
max        741.000000
Name: total_frames, dtype: float64


,video,label,dataset,category,method,target,source,sequence,full_path,total_frames,fps,duration_sec,resolution
41950,id21_id0_0007.mp4,1,Celeb-DF-v3,FaceSwap,Celeb-DF-v2,id0,id21,id0,/home/lucia_pola/internship-deepfake-forensic/...,317,30.0,10.566667,812x456
4231,id4_id28_0002.mp4,1,Celeb-DF-v3,FaceReenact,FSRT,id28,id4,id28,/home/lucia_pola/internship-deepfake-forensic/...,176,20.0,8.800000,256x256
1128,id47_id48_0009.mp4,1,Celeb-DF-v3,FaceReenact,LivePortrait,id48,id47,id48,/home/lucia_pola/internship-deepfake-forensic/...,338,30.0,11.266667,842x476
38041,id28_id29_0003.mp4,1,Celeb-DF-v3,FaceSwap,UniFace,id29,id28,id29,/home/lucia_pola/internship-deepfake-forensic/...,332,30.0,11.066667,846x474
41977,id23_id29_0000.mp4,1,Celeb-DF-v3,FaceSwap,Celeb-DF-v2,id29,id23,id29,/home/lucia_pola/internship-deepfake-forensic/...,318,30.0,10.600000,846x476
37063,id40_id43_0003.mp4,1,Celeb-DF-v3,FaceSwap,UniFace,id43,id40,id43,/home/lucia_pola/internship-deepfake-forensic/...,319,30.0,10.633333,848x478
35629,id54_id51_0003.mp4,1,Celeb-DF-v3,FaceSwap,MobileFaceSwap,id51,id54,id51,/home/lucia_pola/internship-deepfake-forensic/...,314,30.0,10.466667,856x478
30805,id0_0006_test_id02057_1yatisMQIXA.mp4,1,Celeb-DF-v3,TalkingFace,IP_LAP,id02057,id0,0006,/home/lucia_pola/internship-deepfake-forensic/...,130,25.0,5.200000,256x256
11353,id37_id32_0007.mp4,1,Celeb-DF-v3,FaceReenact,LIA,id32,id37,id32,/home/lucia_pola/internship-deepfake-forensic/...,455,30.0,15.166667,256x256
32802,id39_0006_test_id01989_gHVHtKTQBsw.mp4,1,Celeb-DF-v3,TalkingFace,AniTalker,id01989,id39,0006,/home/lucia_pola/internship-deepfake-forensic/...,175,25.0,7.000000,256x256


## 4 - Saving Processed Data

To conclude this notebook, we save the fully cleaned and processed DataFrame to a local CSV file (`./processed_videos/celeb_videos`). This ensures our prepared dataset is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the preprocessing steps.

In [15]:
print("--- SAVING DATAFRAME ---")

output_dir = "./processed_images"
os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, "celeb_videos.csv")
df_celeb.to_csv(save_path, index=False)

print(f"Data succesfully saved to: {save_path}")

--- SAVING DATAFRAME ---
Data succesfully saved to: ./processed_images/celeb_videos.csv
